# 🔱 VoiceBatch Studio v2.2.4 - [Language & Download Fix]
इसमें Hindi Language Lock और Project Name Download फिक्स शामिल है।

In [ ]:
# @title 💤 Step 1: सेटअप और एंटी-स्लीप
import os
from IPython.display import display, Javascript

display(Javascript('''
function ClickConnect(){ document.querySelector("colab-connect-button").click() }
setInterval(ClickConnect,60000)
'''))

print("⏳ लाइब्रेरी तैयार हो रही हैं...")
!pip install -q gradio librosa soundfile coqui-tts torchcodec
os.makedirs("outputs", exist_ok=True)
print("✅ इंजन तैयार है!")

In [ ]:
# @title 🚀 Step 2: ऐप लॉन्च करें (Language Enforcement Fix)
app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import librosa, soundfile as sf
import re, os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def clean_hindi_text(text):
    # केवल देवनागरी और जरूरी निशानों को अनुमति दें ताकि दूसरी भाषा मिक्स न हो
    return re.sub(r'[^\u0900-\u097F\s।?!,:;0-9\[\]]', '', text)

def studio_pro_engine(text, audio_sample, speed, pitch, lang, sil_rem):
    if not audio_sample: return None
    
    # Language Enforcement: अगर हिंदी है तो टेक्स्ट को साफ करें
    if lang == 'hi':
        text = clean_hindi_text(text)
        text = text.replace('...', '। ')
    
    # प्रोजेक्ट का नाम फाइल में सेट करना
    out_filename = 'VoiceBatch_Studio_Output.wav'
    out_path = os.path.join('outputs', out_filename)
    
    tts.tts_to_file(
        text=text, 
        speaker_wav=audio_sample, 
        language=lang, 
        file_path=out_path,
        split_sentences=True
    )
    
    y, sr = librosa.load(out_path)
    if sil_rem: y, _ = librosa.effects.trim(y, top_db=25)
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    sf.write(out_path, y, sr)
    return out_path

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ VoiceBatch Studio v2.2.4')
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label='Script (Hindi Only)', placeholder='यहाँ अपनी हिंदी कहानी लिखें...', lines=8)
            smp = gr.Audio(label='Voice Sample', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en', 'mr'], label='Language', value='hi')
            with gr.Row():
                spd = gr.Slider(0.7, 1.4, 1.0, step=0.01, label="Speed")
                ptc = gr.Slider(-4, 4, 0, step=1, label="Pitch")
            sil = gr.Checkbox(label="Silence Remover", value=True)
            btn = gr.Button('Generate High Quality Audio 🔱', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Download Project Audio')
            gr.Markdown('**Note:** अब फाइल का नाम "VoiceBatch_Studio_Output.wav" होगा।')

    btn.click(studio_pro_engine, [txt, smp, spd, ptc, lng, sil], out)

demo.launch(share=True, debug=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
print("✅ ऐप तैयार है! लॉन्च हो रहा है...")
!python app.py

In [ ]:
# @title 📁 Step 3: डाउनलोड (Direct Download)
from google.colab import files
if os.path.exists('outputs/VoiceBatch_Studio_Output.wav'):
    files.download('outputs/VoiceBatch_Studio_Output.wav')
    print("✅ VoiceBatch Studio फाइल डाउनलोड हो रही है।")
else:
    print("⚠️ ऑडियो फाइल नहीं मिली!")